## **04_nn_module: Encapsulating the Logic in an `nn.Module`**

The raw tensor walkthrough is fantastic for understanding the mechanics.  
In practice, we package this logic into a reusable class.  
This makes our code clean, organized, and easy to stack.

Here is the complete, encapsulated code for a single attention head.  
By the end of this section, every single line will be crystal clear.  

The key optimization: instead of three separate `nn.Linear` layers for Q, K, and V,  
we use **one larger, much faster** fused linear layer.

| Our Manual Walkthrough (Conceptually Clear) | Fused Layer (Computationally Efficient) |
| :--- | :--- |
| `q_proj = nn.Linear(C, C)` | |
| `k_proj = nn.Linear(C, C)` | `c_attn = nn.Linear(C, 3*C)` |
| `v_proj = nn.Linear(C, C)` | |

Instead of three smaller matrix multiplications, the GPU can perform one larger, faster one.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
from dataclasses import dataclass

class SingleHeadSelfAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd, bias=False)

    def forward(self, x):
        B, T, C = x.size()

        # 1. Get Q, K, V from a single projection and split them
        qkv = self.c_attn(x)
        q, k, v = qkv.split(C, dim=2)

        # 2. Calculate attention weights
        # (B, T, C) @ (B, C, T) -> (B, T, T)
        scaled_scores = (q @ k.transpose(-2, -1)) / math.sqrt(k.size(-1))
        attention_weights = F.softmax(scaled_scores, dim=-1)

        # 3. Aggregate values
        # (B, T, T) @ (B, T, C) -> (B, T, C)
        output = attention_weights @ v

        return output

print("Module defined successfully.")

### Breaking down the `__init__`: The Fused Linear Layer

```python
self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd, bias=False)
```

This single line replaces our three separate projection layers.  
The `bias=False` is a common simplification used in minimal implementations like NanoGPT.

### Breaking down the `forward`: Projection and Splitting

```python
qkv = self.c_attn(x)           # (B, T, C) -> (B, T, 3*C)
q, k, v = qkv.split(C, dim=2)  # Three tensors, each (B, T, C)
```

1. `self.c_attn(x)` produces a tensor of shape `(B, T, 3*C)`
2. `.split(C, dim=2)` carves it up: "Along dimension 2, create chunks of size `C`"  
   Since total is `3*C`, we get exactly three tensors of shape `(B, T, C)` — our Q, K, and V

### Proof of Equivalence

Let's prove this class is identical to our manual work from `03_self_attention`.  
We'll load the weights from our separate `q_proj`, `k_proj`, `v_proj` layers into the single `c_attn` layer.

In [ ]:
B, T, C = 1, 4, 2
x = torch.tensor([
    [[0.1, 0.1],   # A
     [1.0, 0.2],   # crane
     [0.1, 0.9],   # ate
     [0.8, 0.0]]   # fish
]).float()

# --- Manual approach (from 03_self_attention) ---
q_proj = nn.Linear(C, C, bias=False)
k_proj = nn.Linear(C, C, bias=False)
v_proj = nn.Linear(C, C, bias=False)

torch.manual_seed(42)
q_proj.weight.data = torch.randn(C, C)
k_proj.weight.data = torch.randn(C, C)
v_proj.weight.data = torch.randn(C, C)

q = q_proj(x)
k = k_proj(x)
v = v_proj(x)

scores = q @ k.transpose(-2, -1)
d_k = k.size(-1)
scaled_scores = scores / math.sqrt(d_k)
attention_weights = F.softmax(scaled_scores, dim=-1)
manual_output = attention_weights @ v

print("Manual output:", manual_output)

In [ ]:
# --- Fused approach (our new module) ---
@dataclass
class GPTConfig:
    n_embd: int

model = SingleHeadSelfAttention(GPTConfig(n_embd=C))

# The c_attn layer's weight is shape (3*C, C)
# We concatenate our separate weights along dim=0 to get (3*C, C)
model.c_attn.weight.data = torch.cat(
    [q_proj.weight.data, k_proj.weight.data, v_proj.weight.data], dim=0
)

model_output = model(x)

print("Module output:", model_output)
print("\nAre the outputs the same?", torch.allclose(manual_output, model_output))

It works perfectly. We have successfully implemented the core of self-attention and formalized it in a clean, reusable module.

However, our model has a flaw — a big one.  
For language generation, **tokens can see into the future**.  
In our current setup, the word "A" can see "crane", "ate", and "fish".  
When generating text one word at a time, that's **cheating**.  

We will fix this next by adding a **causal mask**.